
# 01 | Database Setup

**Objective:** Set up a SQL database for exploring historical FIFA World Cup performance, with a focus on Spain, Argentina, and other competing nations.

This notebook covers database creation, source data inspection, and preparation for SQL analysis.
  


## 1.1 | Importing Libraries

Importing the Python libraries needed to manage file paths, inspect datasets, and connect to SQLite.
  

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path



## 1.2 | Defining Project Paths

Setting up relative paths to keep the project organized and portable across computers.


In [2]:

# Project directories
project_dir = Path.cwd().parent

data_dir = project_dir / "Data"
original_dir = data_dir / "Original"
prepared_dir = data_dir / "Prepared"

# Creating folders if needed
original_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)

# Displaying available source files
list(original_dir.iterdir())


[PosixPath('/Users/raulbrand/Desktop/Personal/Project Folder/world-cup-2026-analysis/Data/Original/.DS_Store'),
 PosixPath('/Users/raulbrand/Desktop/Personal/Project Folder/world-cup-2026-analysis/Data/Original/csv')]


## 1.3 | Inspecting Source Datasets

I start by reviewing the available datasets to understand the files I'll be working with before importing them into SQL.

In [3]:

# Locate the source CSV files
csv_dir = original_dir / "csv"

csv_files = sorted(csv_dir.glob("*.csv"))

# Display available datasets
for file in csv_files:
    print(file.name)

print(f"\nTotal CSV files: {len(csv_files)}")


attempts_at_goal.csv
match_appearances.csv
match_teams.csv
matches.csv
passing_network_edges.csv
player_crosses_open_play.csv
player_events.csv
player_in_possession_distributions.csv
player_line_breaks.csv
player_offers_receptions.csv
player_out_of_possession.csv
player_physical_data.csv
players.csv
team_aerial_control.csv
team_defensive_pressure.csv
team_goal_prevention.csv
team_goalkeeping_distribution.csv
team_key_stats.csv
team_phases.csv
team_set_plays.csv
teams.csv

Total CSV files: 21



## 1.4 | Reviewing Dataset Structure

I review the size and structure of each dataset to identify the available information and determine which tables will support my analysis.


In [4]:

# Review dataset dimensions and columns
dataset_summary = []

for file in csv_files:
    df = pd.read_csv(file)

    dataset_summary.append({
        "Dataset": file.stem,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column Names": ", ".join(df.columns)
    })

# Display dataset overview
summary_df = pd.DataFrame(dataset_summary)

display(summary_df)


,Dataset,Rows,Columns,Column Names
0,attempts_at_goal,2571,14,"appearance_id, body_part, delivery_type, event..."
1,match_appearances,5392,19,"appearance_id, appeared, goal_minutes, is_star..."
2,match_teams,208,6,"home_away, match_id, match_team_id, score, tea..."
3,matches,104,15,"away_score, away_team, away_team_id, date, gro..."
4,passing_network_edges,52072,15,"edge_id, from_appearance_id, from_player, from..."
5,player_crosses_open_play,3288,16,"appearance_id, cutback, driven, inswing, lofte..."
6,player_events,2587,13,"appearance_id, event_id, event_type, match_id,..."
7,player_in_possession_distributions,3288,23,"appearance_id, attempts_at_goal, ball_progress..."
8,player_line_breaks,3288,27,"appearance_id, direction_around, direction_ove..."
9,player_offers_receptions,3288,17,"appearance_id, in_behind, in_between, in_front..."



## 1.5 | Exploring Match Data

I examine the match dataset to understand its structure, identify the tournaments represented, and confirm the available information for team-level analysis.


In [5]:

# Load match data
matches_df = pd.read_csv(csv_dir / "matches.csv")

# Review column names and data types
print("MATCH DATA STRUCTURE")
print(matches_df.dtypes)

# Preview match records
display(matches_df.head(10))


MATCH DATA STRUCTURE
away_score        int64
away_team           str
away_team_id        str
date                str
group               str
home_score        int64
home_team           str
home_team_id        str
kickoff             str
match_id            str
match_number      int64
page_count        int64
penalty_result      str
source_url          str
venue               str
dtype: object


,away_score,away_team,away_team_id,date,group,home_score,home_team,home_team_id,kickoff,match_id,match_number,page_count,penalty_result,source_url,venue
0,0,South Africa,RSA,11 June 2026,Group A,2,Mexico,MEX,13:00,2026-M001-MEX-RSA,1,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Mexico City Stadium
1,1,Czechia,CZE,11 June 2026,Group A,2,Korea Republic,KOR,20:00,2026-M002-KOR-CZE,2,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Guadalajara Stadium
2,1,Bosnia and Herzegovina,BIH,12 June 2026,Group B,1,Canada,CAN,15:00,2026-M003-CAN-BIH,3,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Toronto Stadium
3,1,Paraguay,PAR,12 June 2026,Group D,4,USA,USA,18:00,2026-M004-USA-PAR,4,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Los Angeles Stadium
4,1,Scotland,SCO,13 June 2026,Group C,0,Haiti,HAI,21:00,2026-M005-HAI-SCO,5,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Boston Stadium
5,0,Türkiye,TUR,13 June 2026,Group D,2,Australia,AUS,21:00,2026-M006-AUS-TUR,6,53,NaN,https://www.fifatrainingcentre.com/media/nativ...,BC Place Vancouver
6,1,Morocco,MAR,13 June 2026,Group C,1,Brazil,BRA,18:00,2026-M007-BRA-MAR,7,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,New York/New Jersey Stadium
7,1,Switzerland,SUI,13 June 2026,Group B,1,Qatar,QAT,12:00,2026-M008-QAT-SUI,8,53,NaN,https://www.fifatrainingcentre.com/media/nativ...,San Francisco Bay Area Stadium
8,0,Ecuador,ECU,14 June 2026,Group E,1,Côte d'Ivoire,CIV,19:00,2026-M009-CIV-ECU,9,52,NaN,https://www.fifatrainingcentre.com/media/nativ...,Philadelphia Stadium
9,1,Curaçao,CUW,14 June 2026,Group E,7,Germany,GER,12:00,2026-M010-GER-CUW,10,53,NaN,https://www.fifatrainingcentre.com/media/nativ...,Houston Stadium



## 1.6 | Validating Match Coverage

I check the match dates, team coverage, and source information to understand the scope of the dataset before using it for performance analysis.


In [6]:

# Convert match dates
matches_df["date_parsed"] = pd.to_datetime(
    matches_df["date"], dayfirst=True, errors="coerce"
)

# Review match coverage
print("First match:", matches_df["date_parsed"].min())
print("Last match:", matches_df["date_parsed"].max())
print("Total matches:", len(matches_df))

# Count unique participating teams
teams = pd.concat([
    matches_df["home_team"],
    matches_df["away_team"]
])

print("Unique teams:", teams.nunique())

# Review source and match identifiers
display(matches_df[
    ["match_id", "date", "home_team", "away_team",
     "home_score", "away_score", "source_url"]
].tail(5))


First match: 2026-06-11 00:00:00
Last match: 2026-07-19 00:00:00
Total matches: 104
Unique teams: 48


,match_id,date,home_team,away_team,home_score,away_score,source_url
99,2026-M100-ARG-SUI,11 July 2026,Argentina,Switzerland,3,1,https://www.fifatrainingcentre.com/media/nativ...
100,2026-M101-FRA-ESP,14 July 2026,France,Spain,0,2,https://www.fifatrainingcentre.com/media/nativ...
101,2026-M102-ENG-ARG,15 July 2026,England,Argentina,1,2,https://www.fifatrainingcentre.com/media/nativ...
102,2026-M103-FRA-ENG,18 July 2026,France,England,4,6,https://www.fifatrainingcentre.com/media/nativ...
103,2026-M104-ESP-ARG,19 July 2026,Spain,Argentina,1,0,https://www.fifatrainingcentre.com/media/nativ...



## 1.7 | Creating the SQL Database

I create a SQLite database to store the World Cup datasets and prepare them for SQL-based analysis.

In [7]:

# Define database location
db_path = prepared_dir / "world_cup_2026.db"

# Create database connection
conn = sqlite3.connect(db_path)

print("Database created successfully!")
print(f"Location: {db_path}")


Database created successfully!
Location: /Users/raulbrand/Desktop/Personal/Project Folder/world-cup-2026-analysis/Data/Prepared/world_cup_2026.db



## 1.8 | Importing Match Data into SQL

I load the match dataset into SQLite to begin querying match results, team performance, and tournament statistics.


In [8]:

# Import match data into SQLite
matches_df.drop(columns=["date_parsed"]).to_sql(
    "matches",
    conn,
    if_exists="replace",
    index=False
)

# Confirm imported records
query = """
SELECT COUNT(*) AS total_matches
FROM matches;
"""

display(pd.read_sql_query(query, conn))


,total_matches
0,104



## 1.9 | Importing Remaining Datasets

I import the remaining datasets into SQLite, using each CSV filename as its corresponding table name.


In [9]:

# Import remaining CSV datasets
for file in csv_files:
    if file.stem == "matches":
        continue

    df = pd.read_csv(file)

    df.to_sql(
        file.stem,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"Imported: {file.stem} ({len(df):,} rows)")


Imported: attempts_at_goal (2,571 rows)
Imported: match_appearances (5,392 rows)
Imported: match_teams (208 rows)
Imported: passing_network_edges (52,072 rows)
Imported: player_crosses_open_play (3,288 rows)
Imported: player_events (2,587 rows)
Imported: player_in_possession_distributions (3,288 rows)
Imported: player_line_breaks (3,288 rows)
Imported: player_offers_receptions (3,288 rows)
Imported: player_out_of_possession (3,288 rows)
Imported: player_physical_data (3,288 rows)
Imported: players (1,277 rows)
Imported: team_aerial_control (208 rows)
Imported: team_defensive_pressure (208 rows)
Imported: team_goal_prevention (208 rows)
Imported: team_goalkeeping_distribution (832 rows)
Imported: team_key_stats (3,432 rows)
Imported: team_phases (3,536 rows)
Imported: team_set_plays (3,328 rows)
Imported: teams (48 rows)



## 1.10 | Verifying SQL Tables

I verify that all datasets were imported successfully and are available for SQL analysis.


In [10]:

# Retrieve database tables
query = """
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tables_df = pd.read_sql_query(query, conn)

display(tables_df)

print(f"Total SQL tables: {len(tables_df)}")


,table_name
0,attempts_at_goal
1,match_appearances
2,match_teams
3,matches
4,passing_network_edges
5,player_crosses_open_play
6,player_events
7,player_in_possession_distributions
8,player_line_breaks
9,player_offers_receptions


Total SQL tables: 21



## 1.11 | Exploring Table Relationships

I examine the structure of the main tables to identify shared identifiers and understand how match, team, and player data can be connected.


In [11]:

# Inspect the structure of key SQL tables
tables = ["matches", "match_teams", "teams", "players", "match_appearances"]

for table in tables:
    print(f"\nTABLE: {table}")

    query = f"PRAGMA table_info({table});"

    display(pd.read_sql_query(query, conn)[
        ["name", "type", "pk"]
    ])



TABLE: matches


,name,type,pk
0,away_score,INTEGER,0
1,away_team,TEXT,0
2,away_team_id,TEXT,0
3,date,TEXT,0
4,group,TEXT,0
5,home_score,INTEGER,0
6,home_team,TEXT,0
7,home_team_id,TEXT,0
8,kickoff,TEXT,0
9,match_id,TEXT,0



TABLE: match_teams


,name,type,pk
0,home_away,TEXT,0
1,match_id,TEXT,0
2,match_team_id,TEXT,0
3,score,INTEGER,0
4,team_id,TEXT,0
5,team_name,TEXT,0



TABLE: teams


,name,type,pk
0,team_id,TEXT,0
1,team_name,TEXT,0



TABLE: players


,name,type,pk
0,appearances_count,INTEGER,0
1,display_name,TEXT,0
2,player_id,TEXT,0
3,shirt_numbers,INTEGER,0
4,team_id,TEXT,0



TABLE: match_appearances


,name,type,pk
0,appearance_id,TEXT,0
1,appeared,INTEGER,0
2,goal_minutes,TEXT,0
3,is_starter,INTEGER,0
4,match_id,TEXT,0
5,match_team_id,TEXT,0
6,own_goal_minutes,REAL,0
7,player_id,TEXT,0
8,player_name,TEXT,0
9,position,TEXT,0



## 1.12 | Validating Database Relationships

I check the uniqueness of key identifiers and verify that records across the main tables can be connected consistently.


In [12]:

# Validate key identifiers and table relationships
query = """
SELECT
    (SELECT COUNT(*) FROM matches) AS total_matches,
    (SELECT COUNT(DISTINCT match_id) FROM matches) AS unique_matches,
    (SELECT COUNT(*) FROM teams) AS total_teams,
    (SELECT COUNT(DISTINCT team_id) FROM teams) AS unique_teams,
    (SELECT COUNT(*) FROM players) AS total_players,
    (SELECT COUNT(DISTINCT player_id) FROM players) AS unique_players,
    (SELECT COUNT(*) FROM match_appearances) AS appearances;
"""

display(pd.read_sql_query(query, conn))


,total_matches,unique_matches,total_teams,unique_teams,total_players,unique_players,appearances
0,104,104,48,48,1277,1277,5392


In [13]:

# Check for unmatched relationships
query = """
SELECT
    (SELECT COUNT(*)
     FROM match_teams mt
     LEFT JOIN matches m ON mt.match_id = m.match_id
     WHERE m.match_id IS NULL) AS unmatched_match_teams,

    (SELECT COUNT(*)
     FROM match_appearances ma
     LEFT JOIN players p ON ma.player_id = p.player_id
     WHERE p.player_id IS NULL) AS unmatched_players,

    (SELECT COUNT(*)
     FROM match_appearances ma
     LEFT JOIN matches m ON ma.match_id = m.match_id
     WHERE m.match_id IS NULL) AS unmatched_matches;
"""

display(pd.read_sql_query(query, conn))

,unmatched_match_teams,unmatched_players,unmatched_matches
0,0,0,0
